# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to explore the [FAIR^2 dataset](https://doi.org/10.71728/senscience.y7m0-f273) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, starting from the linked Croissant schema.

### Dataset Source
The dataset is defined by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# If not already installed, install mlcroissant (uncomment the next line if running in Colab or a fresh environment)
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset loaded!")
print(f"Title: {getattr(metadata, 'name', '')}\n\nDescription: {getattr(metadata, 'description', '')}\n")

## 2. Data Overview
Let's list all available `RecordSet` objects, along with their `@id`s and available fields/columns.

In [ ]:
# Helper to pretty-print record set, field, and column @id information
def extract_recordsets_info(dataset):
    info = []
    if getattr(dataset.metadata, 'recordSet', None):
        for rs in dataset.metadata.recordSet:
            rs_id = getattr(rs, '@id', None)
            rs_name = getattr(rs, 'name', None)
            fields = []
            if hasattr(rs, 'field') and rs.field:
                for f in rs.field:
                    fid = getattr(f, '@id', None)
                    fname = getattr(f, 'name', None)
                    ftype = getattr(f, 'dataType', None)
                    fields.append({'@id': fid, 'name': fname, 'type': ftype})
            columns = []
            # Some RecordSets may be tabular, with columns
            if hasattr(rs, 'column') and rs.column:
                for c in rs.column:
                    cid = getattr(c, '@id', None)
                    cname = getattr(c, 'name', None)
                    ctype = getattr(c, 'dataType', None)
                    columns.append({'@id': cid, 'name': cname, 'type': ctype})
            info.append({'@id': rs_id, 'name': rs_name, 'fields': fields, 'columns': columns})
    return info

recordsets_info = extract_recordsets_info(dataset)

if recordsets_info:
    print("Available Record Sets:")
    for rs in recordsets_info:
        print(f"- RecordSet Name: {rs['name']}   @id: {rs['@id']}")
        if rs['fields']:
            print("  Fields (@id, name, type):")
            for f in rs['fields']:
                print(f"    {f['@id']}	'{f['name']}'	({f['type']})")
        if rs['columns']:
            print("  Columns (@id, name, type):")
            for c in rs['columns']:
                print(f"    {c['@id']}	'{c['name']}'	({c['type']})")
        print()
else:
    print("No record sets found in metadata.\nThis Croissant package may only define file distributions or high-level metadata.\n\nTo check for raw data distributions, see metadata.distribution for DataDownload URLs:")
    if hasattr(metadata, 'distribution'):
        for d in metadata.distribution:
            print(f"- distribution @id: {getattr(d, '@id', None)}")
    else:
        print("No distribution found.")

## 3. Data Extraction
Now we'll try to extract data from regular `RecordSet` objects as DataFrames. This is useful if the schema describes structured/tabular data. All entities are referenced by their `@id` as required.

> If no RecordSets exist, you may need to access distributions or files manually.

In [ ]:
# Collect all RecordSet @ids for extraction
record_set_ids = [rs['@id'] for rs in recordsets_info] if recordsets_info else []
dataframes = {}
print("Extracting record sets:")
for rs_id in record_set_ids:
    print(f"- Extracting {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"  Loaded {len(df)} rows.")
    except Exception as e:
        print(f"  Failed to load: {e}")
if dataframes:
    rs0 = record_set_ids[0]
    print(f"\nFields/columns in {rs0}:")
    print(dataframes[rs0].columns.tolist())
    dataframes[rs0].head()
else:
    print("No tabular record sets found.\nIf your dataset is file-based, you may need to download and inspect files via metadata.distribution.")

## 4. Exploratory Data Analysis (EDA)
Let's examine numeric and categorical fields in a selected RecordSet for basic exploratory data analysis.

We'll:
- Select a numeric field by `@id` (if available)
- Filter records based on a threshold
- Normalize that field
- Group by a categorical field

All columns/fields are referenced using their `@id` (not display names).

In [ ]:
# Select the first available record set and check for numeric fields
if dataframes:
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id]
    print(f"Working with RecordSet: {rs_id}")
    # Try to find numeric columns by heuristic or use field type info from metadata
    numeric_cols = []
    categorical_cols = []
    # Map field type info if present
    rs_info = next((rs for rs in recordsets_info if rs['@id'] == rs_id), {})
    field_types = {}
    for f in rs_info.get('fields', [])+rs_info.get('columns', []):
        if f.get('type') in ['schema:Float', 'schema:Integer', 'Float', 'Integer']:
            numeric_cols.append(f['@id'])
        elif f.get('type') in ['schema:Text', 'Text', 'schema:DefinedTerm', 'DefinedTerm']:
            categorical_cols.append(f['@id'])
        field_types[f['@id']] = f.get('type')
    # Fall back to pandas dtype check if types not provided
    if not numeric_cols:
        numeric_cols = df.select_dtypes(include='number').columns.tolist()
    if not categorical_cols:
        categorical_cols = df.select_dtypes(include='object').columns.tolist()

    print(f"Numeric columns by '@id': {numeric_cols}")

    if numeric_cols:
        # Using the first numeric field
        numeric_field_id = numeric_cols[0]
        # Sanitize for possible string conversion or missing data
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean()  # Use mean as sensible default
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.3f} (mean): {len(filtered_df)} rows")

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' (first 5 records):")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Choose a group/categorical field
        group_field_id = None
        for col in categorical_cols:
            if col in filtered_df.columns:
                group_field_id = col
                break

        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
            print(grouped_df.head())
        else:
            print("No categorical columns for grouping found.")
    else:
        print("No numeric fields found for EDA.")
else:
    print("No loaded records available for EDA.")

## 5. Visualization
Visualize numeric field distributions or relationships between fields, using field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Histogram and boxplot of the first numeric field
if dataframes and 'numeric_field_id' in locals():
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.title(f"Histogram of {numeric_field_id}")
    plt.xlabel(numeric_field_id)

    plt.subplot(1,2,2)
    sns.boxplot(x=df[numeric_field_id].dropna())
    plt.title(f"Boxplot of {numeric_field_id}")
    plt.xlabel(numeric_field_id)

    plt.tight_layout()
    plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we loaded metadata from the FAIR^2 dataset defined by a Croissant schema, inspected available record sets and their fields by `@id`, and attempted programmatic data loading and analysis using `mlcroissant`. All data elements were referenced by their global `@id`, ensuring reproducibility and consistency with the FAIR principles.

- If the dataset described `RecordSet` objects, we explored fields and performed basic EDA.
- If the dataset only defined metadata and data files, we located distribution endpoints and described next steps for file-based inspection.

To repeat or extend this analysis:
- Refer to all entities (record set, fields, columns) using their `@id` as shown above.
- Use the `mlcroissant` API to access relationships, provenance, or download linked distributions as needed.

For more, see the [mlcroissant documentation](https://github.com/mlcommons/croissant) and [Croissant schema specification](https://mlcommons.org/croissant/).